In [4]:
import sys
from pathlib import Path

# ── Locate project root and register the basis_data package ───────────────────
# Works whether the kernel CWD is notebooks/ or the project root.
for _candidate in [Path.cwd(), Path.cwd().parent]:
    if (_candidate / "basis_data" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from basis_data.pipeline import build_bond_history

bond_history, cds_mappings_filtered = build_bond_history()
bond_history.head(5)

[cache] cds_transform_interpolated_mapped_tickers_2025.parquet is current — loading from disk
cds_transform_interpolated (2708746, 14)
bond_history (raw) (119770, 5)
bond_history (final) (119770, 21)


,security,date,LAST_PRICE,Benchmark_Spread,G_Spread,MATURITY,SECURITY_NAME,target_cds_maturity,ticker,cds_index,...,BOND_ISIN,bbg_cds_ticker,cds_spread,upfront,cds_spread_5y,cds_5y_maturity,cds_spread_5y_fixed,basis_spread,basis_spread_5y_fixed,basis_price
0,BE6364524635 Corp,2026-07-23,98.071,66.593,65.741,2033-05-19,ABIBB 3 3/8 05/19/33,2033-06-20,ABIBB,Main,...,BE6364524635,ABIB CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
1,DE000A4DFLQ6 Corp,2026-07-23,103.776,149.997,147.453,2031-04-01,SHAEFF 5 3/8 04/01/31,2031-06-20,SHAEFF,XO,...,DE000A4DFLQ6,SCHAAG CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
2,FI4000590864 Corp,2026-07-23,96.108,187.510,183.239,2031-05-28,METSA 3 7/8 05/28/31,2031-06-20,METSA,XO,...,FI4000590864,METSA CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
3,FR0011270487 Corp,2026-07-23,84.706,430.197,430.288,2032-06-14,FTI 4 06/14/32,2032-06-20,FTI,XO_Legacy,...,FR0011270487,FTI CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN
4,FR0014004R72 Corp,2026-07-23,88.265,82.045,78.609,2030-07-27,ALOFP 0 1/2 07/27/30,2030-12-20,ALOFP,Main,...,FR0014004R72,ALSTOM CDS EUR SR 5Y D14 Corp,NaN,NaN,NaN,2031-06-20,NaN,NaN,NaN,NaN


In [5]:
import pandas as pd

# Tickers to drop from the summary entirely (e.g. stale/broken proxies),
# applied before the groupby below.
EXCLUDE_TICKERS = ["CSCHLD", "DISH"]


def _last_valid(series):
    """Last non-NaN value in a series, or None if all NaN."""
    valid = series.notna()
    return series[valid].iloc[-1] if valid.any() else None


def _basis_window_stats(g, cds_col, basis_col):
    """latest/min/max basis plus the number of obs (days with both cds_col
    and G_Spread available) -- generic helper shared by the
    target-maturity-matched basis and the fixed-5Y-maturity basis."""
    valid = g[cds_col].notna() & g["G_Spread"].notna()
    valid_dates = g.loc[valid, "date"]
    if valid_dates.empty:
        return None, None, None, 0
    date_end = valid_dates.iloc[-1]
    latest_basis = g.loc[g["date"] == date_end, basis_col].iloc[-1]
    basis_valid = g.loc[valid, basis_col]
    return latest_basis, basis_valid.min(), basis_valid.max(), len(basis_valid)


def _summarize_ticker(g):
    """One row of cash-vs-CDS basis summary stats for a single CDS ticker's
    proxy bond. date_start/date_end bound the window where both the bond's
    G-spread and the maturity-matched CDS spread are available, since basis
    is only computable there; latest_CDS/latest_bond_g/obs/basis/min/max/
    range_pct are all computed over that same window (bond G-spread vs. the
    CDS spread at the bond's own target maturity; obs is that window's
    size -- the number of days with both a bond G-spread and CDS spread).
    basis_5y/min_5y/max_5y are the same stats but for the bond's G-spread vs.
    the CDS spread at the fixed 5Y maturity (cds_spread_5y_fixed) instead of
    the bond's own target maturity, since 5Y is the reliably liquid point --
    these are the primary basis columns (rows are exported only when
    range_5y_pct resolves). latest_5y and latest_benchmark are informational
    reference stats (like the equivalent chart legend entries), so they're
    just the last available value in the ticker's full history, independent
    of either basis window. range_pct is the latest basis's position within
    [min, max]: 100% at the historical max, 0% at the min."""
    ticker = g.name  # captured before sort_values, which drops the groupby name
    g = g.sort_values("date")
    valid = g["cds_spread"].notna() & g["G_Spread"].notna()
    valid_dates = g.loc[valid, "date"]

    if valid_dates.empty:
        date_start = date_end = pd.NaT
        latest_cds_spread = latest_bond_g = None
    else:
        date_start = valid_dates.iloc[0]
        date_end = valid_dates.iloc[-1]
        last_row = g.loc[g["date"] == date_end].iloc[-1]
        latest_cds_spread = last_row["cds_spread"]
        latest_bond_g = last_row["G_Spread"]

    basis, min_basis, max_basis, obs = _basis_window_stats(g, "cds_spread", "basis_spread")
    range_pct = None
    if basis is not None:
        basis_range = max_basis - min_basis
        range_pct = 100.0 if basis_range == 0 else (basis - min_basis) / basis_range * 100

    basis_5y, min_5y, max_5y, _ = _basis_window_stats(g, "cds_spread_5y_fixed", "basis_spread_5y_fixed")
    range_5y_pct = None
    if basis_5y is not None:
        basis_5y_range = max_5y - min_5y
        range_5y_pct = 100.0 if basis_5y_range == 0 else (basis_5y - min_5y) / basis_5y_range * 100

    latest_5y = _last_valid(g["cds_spread_5y"]) if "cds_spread_5y" in g.columns else None
    latest_benchmark = _last_valid(g["Benchmark_Spread"]) if "Benchmark_Spread" in g.columns else None

    return pd.Series({
        "CDS_Ticker": ticker,
        "CDS_Index": g["cds_index"].iloc[0],
        "bond_name": g["SECURITY_NAME"].iloc[0],
        # grouped/hidden block: identifiers and dates, not needed at a glance
        "bbg_cds_ticker": g["bbg_cds_ticker"].iloc[0],
        "bond_ISIN": g["BOND_ISIN"].iloc[0],
        "date_start": date_start,
        "date_end": date_end,
        "cds_mat": g["target_cds_maturity"].iloc[0],
        "latest_CDS": latest_cds_spread,
        "latest_5y": latest_5y,
        "latest_bond_g": latest_bond_g,
        "latest_benchmark": latest_benchmark,
        "obs": obs,
        "basis": basis,
        "min": min_basis,
        "max": max_basis,
        "range_pct": range_pct,
        "basis_5y": basis_5y,
        "min_5y": min_5y,
        "max_5y": max_5y,
        "range_5y_pct": range_5y_pct,
    })


basis_summary = (
    bond_history[~bond_history["ticker"].isin(EXCLUDE_TICKERS)]
    .groupby("ticker")
    .apply(_summarize_ticker)
    .reset_index(drop=True)
    .sort_values(["CDS_Index", "CDS_Ticker"])
    .reset_index(drop=True)
)
print(basis_summary.shape)
basis_summary.head(10)

(368, 21)


,CDS_Ticker,CDS_Index,bond_name,bbg_cds_ticker,bond_ISIN,date_start,date_end,cds_mat,latest_CDS,latest_5y,...,latest_benchmark,obs,basis,min,max,range_pct,basis_5y,min_5y,max_5y,range_5y_pct
0,RAKUGRO,Asia,RAKUTN 1.05 12/02/31,RAKUTN CDS JPY SR 5Y D14 Corp,JP396720DMC4,NaT,NaT,2031-12-20,NaN,210.283600,...,212.700,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAC,Banks,BAC 4.695 04/23/32,BOFA CDS USD SR 5Y D14 Corp,US06051GNA30,2026-04-17,2026-07-22,2032-06-20,63.115601,54.962799,...,73.537,66,-10.106399,-13.040102,0.336200,21.932089,-18.259201,-21.130998,-7.909398,21.720498
2,C,Banks,C 4.846 06/18/32,CINC CDS USD SR 5Y D14 Corp,US17325FBV94,2026-06-12,2026-07-22,2032-06-20,61.978500,52.754002,...,70.968,27,-7.698500,-9.026600,-2.267900,19.650236,-16.922998,-18.137899,-10.801000,16.558774
3,GS,Banks,GS 4.972 06/03/32,GS CDS USD SR 5Y D14 Corp,US38141GF335,2026-05-28,2026-07-22,2032-06-20,62.250099,54.799999,...,88.258,38,-24.502901,-27.817301,-13.320199,22.862500,-31.953001,-35.738802,-21.293000,26.206930
4,JPM,Banks,JPM 4.622 04/23/32,JPMCC CDS USD SR 5Y D14 Corp,US46647PFM32,2026-04-16,2026-07-22,2032-06-20,47.740002,40.599998,...,74.104,67,-27.000998,-28.115500,-14.513599,8.193719,-34.141002,-35.694700,-22.073101,11.406138
5,MS,Banks,MS 4.809 04/16/32,MS CDS USD SR 5Y D14 Corp,US61748UAW27,2026-04-16,2026-07-22,2032-06-20,66.858101,57.467300,...,83.107,67,-18.379899,-24.095297,-8.768101,37.289258,-27.770700,-33.115900,-17.910699,35.153761
6,WFC,Banks,WFC 4.844 05/20/32,WELLFARGO CDS USD SR 5Y D14 Corp,US95000U4J91,2026-05-14,2026-07-22,2032-06-20,57.374699,49.133801,...,74.771,47,-18.510301,-20.571702,-10.422500,20.310960,-26.751199,-28.588399,-18.567502,18.333680
7,ENTLN,Euro_NonIndex,ENTLN 4 7/8 11/30/31,ENTLN CDS EUR SR 5Y D14 Corp,XS3229426138,2025-11-14,2026-07-22,2031-12-20,218.067596,202.234802,...,169.388,173,40.053596,-98.354894,40.053596,100.000000,24.220802,-111.608694,24.220802,100.000000
8,SDFGR,Euro_NonIndex,SDFGR 4 1/4 06/19/29,SDFGR CDS EUR SR 5Y D14 Corp,XS2844398482,2025-01-02,2026-07-22,2029-06-20,90.926300,131.511093,...,78.863,396,14.362300,-29.588198,43.843704,59.852050,54.947093,12.713598,95.544204,50.987790
9,APODS,Fin,APODS 6.7 07/29/31,CZ061134 Curncy,US03770DAD57,2026-04-09,2026-07-22,2031-12-20,218.270905,210.112198,...,197.938,70,20.436905,14.680001,64.452590,11.566414,12.278198,5.539602,56.929000,13.112814


In [6]:
import os

from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.worksheet.table import Table, TableStyleInfo

OUTPUT_PATH = Path(r"S:\Structured Credit\Matt Stuff\claude_repos\cds_bond_basis\output\bond_cds_basis_summary.xlsx")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Bloomberg field mnemonic for the bond G-spread, kept consistent with the
# field already pulled via bdh_mm() into bond_history's G_Spread column.
G_SPREAD_FIELD = "BLOOMBERG_MID_G_SPREAD"

# live_basis is clamped to this range -- a handful of illiquid/distressed
# names can otherwise print basis values in the thousands that blow out the
# color scale and column width for everything else.
LIVE_BASIS_CAP = 500
# Fixed anchor (not the column's observed min/max, unlike range_pct) for the
# basis/basis_5y diverging scale, centered at 0bp.
BASIS_SCALE_CAP = 200

FORMULA_FILL = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
# Light green flags cds_override as a manual-entry cell, distinct from the
# grey formula-driven live columns around it.
OVERRIDE_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
# Very light neutral banding for the static columns -- deliberately grey
# rather than a built-in blue/orange Excel table style, so it doesn't
# compete with the red/white/blue diverging scales sitting in the same rows.
BAND_FILL = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
# CDS_Index category tint -- pale blue for Main, pale amber for the XO family.
INDEX_FILLS = {
    "Main": PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid"),
    "XO": PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid"),
}

FONT_NAME = "Calibri"
FONT_SIZE = 8
HEADER_FONT = Font(name=FONT_NAME, size=FONT_SIZE, bold=True)
BODY_FONT = Font(name=FONT_NAME, size=FONT_SIZE)
CENTER_ALIGN = Alignment(horizontal="center", vertical="center")
# Dark navy for categorical/identifier columns (names, tickers, ISINs) so
# they read as "what this row is" versus the plain-black numeric results.
CATEGORICAL_FONT_COLOR = "1F3864"
CATEGORICAL_COLUMNS = {"CDS_Ticker", "CDS_Index", "bond_name", "bbg_cds_ticker", "bond_ISIN"}
BOLD_COLUMNS = {"CDS_Ticker", "bond_name"}
# Thin visual seam between the static (historical Python pull) columns and
# the live (Bloomberg BDP formula) columns.
SECTION_BORDER_SIDE = Side(style="medium", color="808080")

RANGE_PCT_COLOR_SCALE = dict(
    start_type="num", start_value=0, start_color="F8696B",
    mid_type="num", mid_value=50, mid_color="FFFFFF",
    end_type="num", end_value=100, end_color="5A8AC6",
)
BASIS_COLOR_SCALE = dict(
    start_type="num", start_value=-BASIS_SCALE_CAP, start_color="F8696B",
    mid_type="num", mid_value=0, mid_color="FFFFFF",
    end_type="num", end_value=BASIS_SCALE_CAP, end_color="5A8AC6",
)

# Identifier/date/reference columns, grouped and collapsed since they're rarely needed at a glance.
GROUPED_COLUMNS = [
    "bbg_cds_ticker", "bond_ISIN", "date_start", "date_end", "cds_mat",
    "latest_CDS", "latest_5y", "latest_bond_g", "latest_benchmark",
]
# Separate group: intermediate live lookups feeding spread_live/g_live, not
# usually needed on their own.
LIVE_GROUPED_COLUMNS = {"bond_bench", "bench_price"}


def _iferror(formula, fallback='"--"'):
    """Wrap a '=...' formula string as =IFERROR(..., fallback)."""
    return f"=IFERROR({formula[1:]},{fallback})"


COLUMN_WIDTHS = {
    "CDS_Ticker": 12, "bbg_cds_ticker": 26, "CDS_Index": 10, "bond_name": 28, "bond_ISIN": 16,
    "date_start": 12, "date_end": 12, "cds_mat": 12,
    "latest_CDS": 14, "latest_5y": 12, "latest_bond_g": 14, "latest_benchmark": 16,
    "obs": 8, "basis": 8, "min": 8, "max": 8, "range_pct": 8,
    "basis_5y": 8, "min_5y": 8, "max_5y": 8, "range_5y_pct": 8,
}
DATE_COLUMNS = {"date_start", "date_end", "cds_mat"}
SPREAD_COLUMNS = {
    "latest_CDS", "latest_5y", "latest_bond_g", "latest_benchmark",
    "basis", "min", "max",
    "basis_5y", "min_5y", "max_5y",
}
PERCENT_COLUMNS = {"range_pct", "range_5y_pct"}
INTEGER_COLUMNS = {"obs"}
# basis/basis_5y are the "headline" result columns -- get a diverging scale
# fixed at +/-BASIS_SCALE_CAP centered on 0, on top of the accounting format
# SPREAD_COLUMNS already gives them.
BASIS_SCALE_COLUMNS = {"basis", "basis_5y"}

# Drop tickers with no computable fixed-5Y basis (range_5y_pct is the
# reference column since the 5Y point is the reliably liquid one), rather
# than exporting a row of blanks.
basis_summary_export = basis_summary[basis_summary["range_5y_pct"].notna()].reset_index(drop=True)
print(f"Dropped {len(basis_summary) - len(basis_summary_export)} rows with no basis data")

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    basis_summary_export.to_excel(writer, sheet_name="Basis Summary", index=False)
    ws = writer.sheets["Basis Summary"]

    for cell in ws[1]:
        cell.font = HEADER_FONT
    ws.freeze_panes = "D2"

    last_row = len(basis_summary_export) + 1
    n_static_cols = len(basis_summary_export.columns)
    for i, col in enumerate(basis_summary_export.columns, start=1):
        col_letter = get_column_letter(i)
        ws.column_dimensions[col_letter].width = COLUMN_WIDTHS.get(col, 14)
        if col in GROUPED_COLUMNS:
            ws.column_dimensions[col_letter].outlineLevel = 1
            ws.column_dimensions[col_letter].hidden = True
        if col in DATE_COLUMNS:
            for cell in ws[col_letter][1:]:
                cell.number_format = "yyyy-mm-dd"
        elif col in SPREAD_COLUMNS:
            # accounting style: negatives in red parens, matching the slideshow charts
            for cell in ws[col_letter][1:]:
                cell.number_format = "#,##0.0;[Red](#,##0.0)"
            if col in BASIS_SCALE_COLUMNS:
                ws.conditional_formatting.add(
                    f"{col_letter}2:{col_letter}{last_row}", ColorScaleRule(**BASIS_COLOR_SCALE)
                )
        elif col in PERCENT_COLUMNS:
            for cell in ws[col_letter][1:]:
                cell.number_format = "0.0\\%"
            # range_pct is already normalized 0-100, so anchor the color scale to
            # those fixed values (not the observed min/max in the column) --
            # red at 0 (historical low), white at 50, blue at 100 (historical high)
            ws.conditional_formatting.add(
                f"{col_letter}2:{col_letter}{last_row}", ColorScaleRule(**RANGE_PCT_COLOR_SCALE)
            )
        elif col in INTEGER_COLUMNS:
            for cell in ws[col_letter][1:]:
                cell.number_format = "#,##0"

    # Neutral grey banding on the static columns only (the live block already
    # has its own uniform grey/green fill, which would look muddy with a
    # second banding pattern layered on top).
    for row in range(3, last_row + 1, 2):
        for col_idx in range(1, n_static_cols + 1):
            ws.cell(row=row, column=col_idx).fill = BAND_FILL

    # CDS_Index category tint, overriding the banding fill above for that column.
    cds_index_col_idx = basis_summary_export.columns.get_loc("CDS_Index") + 1
    for row in range(2, last_row + 1):
        cell = ws.cell(row=row, column=cds_index_col_idx)
        fill = next((f for key, f in INDEX_FILLS.items() if str(cell.value).startswith(key)), None)
        if fill is not None:
            cell.fill = fill

    # Static columns referenced by the live formulas below.
    isin_col = get_column_letter(basis_summary_export.columns.get_loc("bond_ISIN") + 1)
    cds_ticker_col = get_column_letter(basis_summary_export.columns.get_loc("bbg_cds_ticker") + 1)
    # live_basis is a bond-G-spread-vs-5Y-CDS basis, so its range check uses
    # the fixed-5Y-maturity historical range ("min_5y"/"max_5y"), not the
    # target-maturity one (min/max).
    min_basis_col = get_column_letter(basis_summary_export.columns.get_loc("min_5y") + 1)
    max_basis_col = get_column_letter(basis_summary_export.columns.get_loc("max_5y") + 1)

    # Live-refresh columns: re-derive the bond's benchmark, its G-spread, and
    # the CDS-vs-bond basis directly in Excel via the Bloomberg Add-in (BDP),
    # rather than from the historical Python pull. Requires the Add-in to
    # resolve. Each column's formula references the live column(s) to its left.
    # Every formula is wrapped in IFERROR(..., "--") so a missing/unresolved
    # BDP lookup shows as a placeholder instead of an #N/A propagating
    # through the dependent formulas to its right. cds_override is a blank,
    # manually-editable cell; cds_5y falls back to live_cds_5y whenever it's
    # empty, and live_basis is computed off cds_5y so a manual override feeds
    # straight through to the basis/range calc.
    live_col_start = n_static_cols + 1
    live_columns = [
        ("bond_live", 8, "#,##0.000"),
        ("bond_bench", 22, None),
        ("bench_price", 14, "#,##0.000"),
        ("spread_live", 8, "#,##0.0;[Red](#,##0.0)"),
        ("g_live", 8, "#,##0.0;[Red](#,##0.0)"),
        ("live_cds_5y", 8, "#,##0.0;[Red](#,##0.0)"),
        ("cds_override", 8, "#,##0.0;[Red](#,##0.0)"),
        ("cds_5y", 8, "#,##0.0;[Red](#,##0.0)"),
        ("live_basis", 8, "#,##0.0;[Red](#,##0.0)"),
        ("live_range_pct", 8, "0.0\\%"),
    ]
    (bond_live_col, bond_bench_col, bench_price_col, spread_live_col,
     g_live_col, live_cds_5y_col, cds_override_col, cds_5y_col,
     live_basis_col, live_range_pct_col) = (
        get_column_letter(live_col_start + offset) for offset in range(len(live_columns))
    )

    for offset, (name, width, _) in enumerate(live_columns):
        col_letter = get_column_letter(live_col_start + offset)
        ws.cell(row=1, column=live_col_start + offset, value=name).font = HEADER_FONT
        ws.column_dimensions[col_letter].width = width
        if name in LIVE_GROUPED_COLUMNS:
            ws.column_dimensions[col_letter].outlineLevel = 1
            ws.column_dimensions[col_letter].hidden = True

    for row in range(2, last_row + 1):
        ws[f"{bond_live_col}{row}"] = _iferror(f'=BDP({isin_col}{row}&" Corp","LAST_PRICE")')
        ws[f"{bond_bench_col}{row}"] = _iferror(f'=BDP({isin_col}{row}&" Corp","YAS_BENCHMARK_BOND_ISIN")')
        # bond_bench is a bare ISIN (from YAS_BENCHMARK_BOND_ISIN), so it needs
        # the same " CORP" suffix as bond_ISIN to resolve as a BDP security.
        ws[f"{bench_price_col}{row}"] = _iferror(f'=BDP({bond_bench_col}{row}&" CORP","LAST_PRICE")')
        ws[f"{spread_live_col}{row}"] = _iferror(
            f'=BDP({isin_col}{row}&" Corp","YAS_YLD_SPREAD",'
            f'"YAS_BOND_PX="&{bond_live_col}{row},"YAS_BNCHMRK_BOND_PX="&{bench_price_col}{row})'
        )
        ws[f"{g_live_col}{row}"] = _iferror(
            f'=BDP({isin_col}{row}&" Corp","{G_SPREAD_FIELD}",'
            f'"YAS_BOND_PX="&{bond_live_col}{row},"YAS_BNCHMRK_BOND_PX="&{bench_price_col}{row})'
        )
        ws[f"{live_cds_5y_col}{row}"] = _iferror(f'=BDP({cds_ticker_col}{row},"LAST_PRICE")')
        # cds_override left blank -- manual entry only, no formula.
        ws[f"{cds_5y_col}{row}"] = _iferror(
            f'=IF(ISBLANK({cds_override_col}{row}),{live_cds_5y_col}{row},{cds_override_col}{row})'
        )
        # clamped to [-LIVE_BASIS_CAP, +LIVE_BASIS_CAP] so a handful of
        # illiquid/distressed names can't blow out the column's scale/color rule
        ws[f"{live_basis_col}{row}"] = _iferror(
            f'=MAX(-{LIVE_BASIS_CAP},MIN({LIVE_BASIS_CAP},{cds_5y_col}{row}-{g_live_col}{row}))'
        )
        ws[f"{live_range_pct_col}{row}"] = _iferror(
            f'=({live_basis_col}{row}-{min_basis_col}{row})'
            f'/({max_basis_col}{row}-{min_basis_col}{row})*100'
        )

    for offset, (name, width, number_format) in enumerate(live_columns):
        col_letter = get_column_letter(live_col_start + offset)
        # light grey flags formula-driven (live BDP pull) columns; cds_override
        # gets the light green "manual entry" fill instead.
        fill = OVERRIDE_FILL if name == "cds_override" else FORMULA_FILL
        for cell in ws[col_letter][1:]:
            cell.fill = fill
            if number_format is not None:
                cell.number_format = number_format

    # Same fixed 0/50/100 red-white-blue scale as range_pct, applied to its live counterpart.
    ws.conditional_formatting.add(
        f"{live_range_pct_col}2:{live_range_pct_col}{last_row}", ColorScaleRule(**RANGE_PCT_COLOR_SCALE)
    )

    # Column indices (not letters) for the categorical/bold font treatment below.
    categorical_col_idxs = {
        basis_summary_export.columns.get_loc(c) + 1 for c in CATEGORICAL_COLUMNS
    }
    bold_col_idxs = {basis_summary_export.columns.get_loc(c) + 1 for c in BOLD_COLUMNS}

    # Calibri 8pt + centered everywhere -- applied before the table is added
    # so the table style doesn't need to fight with per-cell formatting.
    # Categorical/identifier columns get a dark navy font (bold for
    # CDS_Ticker/bond_name) so they read as "what this row is" versus the
    # plain-black numeric result columns.
    last_col_letter = get_column_letter(live_col_start + len(live_columns) - 1)
    for row in ws.iter_rows(min_row=1, max_row=last_row, min_col=1, max_col=live_col_start + len(live_columns) - 1):
        for cell in row:
            is_header = cell.row == 1
            is_categorical = (not is_header) and cell.column in categorical_col_idxs
            is_bold = is_header or ((not is_header) and cell.column in bold_col_idxs)
            cell.font = Font(
                name=FONT_NAME, size=FONT_SIZE, bold=is_bold,
                color=CATEGORICAL_FONT_COLOR if is_categorical else None,
            )
            cell.alignment = CENTER_ALIGN

    # Thin seam between the static (historical) block and the live
    # (Bloomberg BDP formula) block.
    for row in range(1, last_row + 1):
        cell = ws.cell(row=row, column=live_col_start)
        cell.border = Border(left=SECTION_BORDER_SIDE)

    # Real Excel Table (not just autofilter) spanning every column, static +
    # live -- gives sortable/filterable headers and a named range other
    # formulas/pivots can reference. Row striping is handled manually above
    # (BAND_FILL) rather than via the table style, so it stays neutral grey
    # instead of Excel's built-in blue/orange bands.
    table = Table(displayName="BasisSummary", ref=f"A1:{last_col_letter}{last_row}")
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleLight1",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=False,
        showColumnStripes=False,
    )
    ws.add_table(table)

print(f"Wrote {len(basis_summary_export)} rows to {OUTPUT_PATH}")
os.startfile(OUTPUT_PATH)

Dropped 3 rows with no basis data


PermissionError: [Errno 13] Permission denied: 'S:\\Structured Credit\\Matt Stuff\\claude_repos\\cds_bond_basis\\output\\bond_cds_basis_summary.xlsx'